# Batch-of-One Penalty — Full Experiment

This notebook produces every number and figure for the paper. The smoke test already
confirmed the effect exists, so this run is about measuring it properly and explaining it.

Three questions it answers:

1. How large is the per-row latency penalty at batch size one, across nine serving paths?
2. How much of that penalty is the Python API layer rather than tree traversal?
3. Does the penalty shrink as the model gets bigger, which is what a fixed overhead cost
   would predict?

Everything is written to Google Drive after each measurement. If Colab disconnects, run all
the cells again and it continues from where it stopped rather than starting over.

Expected runtime: forty to sixty minutes on a two-core Colab CPU instance.

## 1. Install Libraries

Two to three minutes. Dependency warnings are normal and can be ignored. Versions are
pinned so the paper can state exactly what produced the numbers.

In [ ]:
!pip install -q lightgbm==4.7.0 xgboost==3.4.1 catboost==1.2.10 \
                onnxruntime==1.24.4 onnxmltools skl2onnx 2>&1 | tail -3
print("install finished")

## 2. Pin Threads and Import

Thread limits have to be set before any library loads, so this cell must run before
everything below it. If you ever restart the runtime, start again from this cell.

In [ ]:
import os
for _v in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS"]:
    os.environ[_v] = "1"

import time, json, platform, random, warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import onnxruntime as ort
from onnxmltools.convert import convert_lightgbm, convert_xgboost
from onnxmltools.convert.common.data_types import FloatTensorType
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
print("imports ok")

## 3. Experiment Configuration

Everything adjustable lives here so nothing further down needs editing.

`TIME_BUDGET_S` is the trick that keeps the run short. Rather than pushing a fixed number
of rows through at every batch size, the harness first times a single call and then picks
how many calls to make so each repeat takes about the same wall time. Without this, batch
size one would dominate the whole run.

`RUN_ID` decides which folder the results go in, and it is the only thing that separates
one run from another. Resume works by reading back what is already saved under the current
run id, so re-running with an unchanged id after a complete run measures nothing at all and
leaves the drift log empty. To re-measure from scratch, change the string. Previous runs
stay where they are and become replication evidence.

Set `QUICK` to True for a five minute rehearsal that touches every code path. Set it back
to False for the real run.

In [ ]:
RUN_ID = "primary"        # change this string to start a clean run, see note below
QUICK = False          # True = short rehearsal, False = the real run

BATCHES      = [1, 2, 4, 8, 16, 32, 64, 128, 256, 512, 1024, 4096, 16384]
TREE_COUNTS  = [100, 500, 1000]
FEATURE_SWEEP = [20, 100]        # synthetic data only
THREAD_COUNTS = [1, 2]
REPEATS      = 11
WARMUP_CALLS = 20
TIME_BUDGET_S = 0.15             # target wall time per repeat
MIN_CALLS, MAX_CALLS = 5, 3000
DEPTH = 6

if QUICK:
    BATCHES = [1, 64, 4096]
    TREE_COUNTS = [100]
    FEATURE_SWEEP = [20]
    THREAD_COUNTS = [1]
    REPEATS = 3
    TIME_BUDGET_S = 0.05

print(f"run id        {RUN_ID}")
print(f"batch sizes   {BATCHES}")
print(f"tree counts   {TREE_COUNTS}")
print(f"repeats       {REPEATS}")

## 4. Set Up Storage and Resume

Colab unmounts Drive on its own during long sessions, so nothing is written to Drive
directly. Every result goes to local disk first, and Drive gets a copy after each
configuration finishes. If Drive drops out mid-run, the experiment keeps going and the
notebook tries to remount at the next sync.

A popup will ask you to pick your Google account and press Allow.

Resume works off whichever copy survives. On startup, if Drive holds results from an
earlier session and local disk doesn't, they get pulled back first.

In [ ]:
import shutil

LOCAL_DIR = f"/content/batch_penalty/{RUN_ID}"
DRIVE_DIR = f"/content/drive/MyDrive/batch_penalty/{RUN_ID}"
os.makedirs(LOCAL_DIR, exist_ok=True)

def mount_drive():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(DRIVE_DIR, exist_ok=True)
        return True
    except Exception as e:
        print("Drive unavailable:", type(e).__name__, str(e)[:90])
        return False

DRIVE_OK = mount_drive()

SAVE_DIR    = LOCAL_DIR
RESULTS_CSV = os.path.join(LOCAL_DIR, "results.csv")
CORRECT_CSV = os.path.join(LOCAL_DIR, "correctness.csv")
NOISE_CSV   = os.path.join(LOCAL_DIR, "noise_reference.csv")
FIG_DIR     = os.path.join(LOCAL_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

# pull back anything a previous session left on Drive so resume still works
if DRIVE_OK:
    for _n in ["results.csv", "correctness.csv", "noise_reference.csv"]:
        _d, _l = os.path.join(DRIVE_DIR, _n), os.path.join(LOCAL_DIR, _n)
        if os.path.exists(_d) and not os.path.exists(_l):
            shutil.copy2(_d, _l)
            print("recovered", _n, "from Drive")

def append_row(record, path):
    """Append one row to a local CSV. Never raises: losing the whole run to a
    filesystem hiccup costs far more than losing a single measurement."""
    try:
        df = record if isinstance(record, pd.DataFrame) else pd.DataFrame([record])
        df.to_csv(path, mode="a", header=not os.path.exists(path), index=False)
        return True
    except Exception as e:
        print("  local write failed:", type(e).__name__, str(e)[:90])
        return False

def sync_to_drive(quiet=True):
    """Mirror local results to Drive, remounting once if Drive dropped out."""
    global DRIVE_OK
    if not DRIVE_OK:
        DRIVE_OK = mount_drive()
    if not DRIVE_OK:
        return False
    try:
        os.makedirs(DRIVE_DIR, exist_ok=True)
        for n in os.listdir(LOCAL_DIR):
            src = os.path.join(LOCAL_DIR, n)
            if os.path.isfile(src):
                shutil.copy2(src, os.path.join(DRIVE_DIR, n))
        fd = os.path.join(DRIVE_DIR, "figures")
        os.makedirs(fd, exist_ok=True)
        for n in os.listdir(FIG_DIR):
            shutil.copy2(os.path.join(FIG_DIR, n), os.path.join(fd, n))
        if not quiet:
            print("  synced to Drive")
        return True
    except Exception as e:
        print("  Drive sync failed, continuing locally:", type(e).__name__, str(e)[:90])
        DRIVE_OK = False
        return False

def cpu_model():
    try:
        for line in open("/proc/cpuinfo"):
            if line.startswith("model name"):
                return line.split(":", 1)[1].strip()
    except Exception:
        pass
    return "unknown"

ENV = {"cpu": cpu_model(), "cores": os.cpu_count(),
       "python": platform.python_version(), "lightgbm": lgb.__version__,
       "xgboost": xgb.__version__, "onnxruntime": ort.__version__,
       "started": time.strftime("%Y-%m-%d %H:%M:%S")}
with open(os.path.join(LOCAL_DIR, "environment.json"), "w") as f:
    json.dump(ENV, f, indent=2)

for k, v in ENV.items():
    print(f"{k:12s} {v}")
print("\nresults written to", LOCAL_DIR)
print("mirrored to", DRIVE_DIR if DRIVE_OK else "(Drive not mounted, local only)")

## 5. Datasets

Two sources, each doing a different job.

Online Shoppers Purchasing Intention from UCI is the real e-commerce task. Around 12,300
browsing sessions, seventeen features, and the label is whether the session ended in a
purchase. Every headline number in the paper comes from this one.

Synthetic data is only there for the feature-count sweep, since the number of columns in a
real dataset cannot be varied. Using synthetic data for that one controlled comparison is
honest and needs saying in the paper.

In [ ]:
URL = ("https://archive.ics.uci.edu/static/public/468/"
       "online+shoppers+purchasing+intention+dataset.zip")

def load_shoppers():
    df = pd.read_csv(URL, compression="zip")
    y = df["Revenue"].astype(int).values
    Xdf = df.drop(columns=["Revenue"])
    for c in Xdf.columns:
        if Xdf[c].dtype == object or str(Xdf[c].dtype) == "bool":
            Xdf[c] = pd.factorize(Xdf[c])[0]
    return Xdf.values.astype(np.float32), y

def make_synth(n_features, n=12000):
    from sklearn.datasets import make_classification
    X, y = make_classification(n_samples=n, n_features=n_features,
                               n_informative=max(5, n_features // 2),
                               weights=[0.85], random_state=SEED)
    return X.astype(np.float32), y

def split(X, y, frac=0.8):
    k = int(frac * len(X))
    return X[:k], y[:k], np.ascontiguousarray(X[k:])

try:
    Xs, ys = load_shoppers()
    SHOPPERS_OK = True
except Exception as e:
    print("UCI download failed:", type(e).__name__, "- falling back to synthetic")
    Xs, ys = make_synth(17)
    SHOPPERS_OK = False

print(f"online_shoppers  rows={len(Xs)}  features={Xs.shape[1]}  "
      f"positive rate={ys.mean():.3f}  real={SHOPPERS_OK}")

## 6. Model Training and Serving Path Construction

One function builds every serving path for a given training set and tree count. Nine paths
in total: the sklearn wrapper for each library, the lower level native calls, and an ONNX
export for each. The point of the comparison is that all nine hold the same model, so any
latency difference is the interface, not the maths.

In [ ]:
def build_runners(X_train, y_train, n_trees, threads, tag):
    """Train three GBDTs and return a dict of callables, all holding the same models."""
    lgbm = lgb.LGBMClassifier(n_estimators=n_trees, max_depth=DEPTH, num_leaves=31,
                              n_jobs=threads, verbose=-1,
                              random_state=SEED).fit(X_train, y_train)
    xgbm = xgb.XGBClassifier(n_estimators=n_trees, max_depth=DEPTH, n_jobs=threads,
                             tree_method="hist",
                             random_state=SEED).fit(X_train, y_train)
    cbm = CatBoostClassifier(iterations=n_trees, depth=DEPTH, thread_count=threads,
                             verbose=0, random_seed=SEED).fit(X_train, y_train)

    lb, xb = lgbm.booster_, xgbm.get_booster()
    r = {
        "lgb_sklearn": lambda b: lgbm.predict_proba(b),
        "lgb_booster": lambda b: lb.predict(b),
        "xgb_sklearn": lambda b: xgbm.predict_proba(b),
        "xgb_dmatrix": lambda b: xb.predict(xgb.DMatrix(b)),
        "xgb_inplace": lambda b: xb.inplace_predict(b),
        "cat_sklearn": lambda b: cbm.predict_proba(b),
    }

    so = ort.SessionOptions()
    so.intra_op_num_threads = threads
    so.inter_op_num_threads = threads
    it = [("input", FloatTensorType([None, X_train.shape[1]]))]

    def reg(name, sess):
        i, o = sess.get_inputs()[0].name, sess.get_outputs()[-1].name
        r[name] = lambda b, s=sess, i=i, o=o: s.run([o], {i: b})

    def sess_of(onx):
        return ort.InferenceSession(onx.SerializeToString(), so,
                                    providers=["CPUExecutionProvider"])
    try:
        reg("lgb_onnx", sess_of(convert_lightgbm(lb, initial_types=it, zipmap=False)))
    except Exception as e:
        print("  skipped lgb_onnx:", type(e).__name__)
    try:
        reg("xgb_onnx", sess_of(convert_xgboost(xb, initial_types=it)))
    except Exception as e:
        print("  skipped xgb_onnx:", type(e).__name__)
    try:
        p = os.path.join(SAVE_DIR, f"catboost_{tag}.onnx")
        cbm.save_model(p, format="onnx")
        reg("cat_onnx", ort.InferenceSession(p, so,
                                             providers=["CPUExecutionProvider"]))
    except Exception as e:
        print("  skipped cat_onnx:", type(e).__name__)
    return r

## 7. The Timing Harness

Each chunk is timed individually rather than timing the whole pass, which costs almost
nothing and gives the tail of the distribution as well as the middle. Tail latency is what
actually matters for a request that a user is waiting on, so the paper reports the 95th
percentile alongside the median.

The calibration step runs a few calls, measures how long they take, and picks a call count
that fills the time budget. That keeps a batch-size-one measurement and a batch-size-16384
measurement to roughly the same wall clock cost.

In [ ]:
def make_pool(X_pool, rows):
    reps = rows // len(X_pool) + 1
    return np.ascontiguousarray(np.tile(X_pool, (reps, 1))[:rows])

def calibrate(fn, X_pool, batch):
    """Pick a call count so one repeat takes roughly TIME_BUDGET_S seconds."""
    probe = make_pool(X_pool, batch)
    for _ in range(3):
        fn(probe)
    t0 = time.perf_counter_ns()
    for _ in range(3):
        fn(probe)
    per_call_s = (time.perf_counter_ns() - t0) / 3 / 1e9
    n = int(TIME_BUDGET_S / max(per_call_s, 1e-9))
    return int(np.clip(n, MIN_CALLS, MAX_CALLS))

def measure(fn, X_pool, batch):
    n_calls = calibrate(fn, X_pool, batch)
    pool = make_pool(X_pool, batch * n_calls)
    chunks = [pool[i * batch:(i + 1) * batch] for i in range(n_calls)]

    for c in chunks[:min(WARMUP_CALLS, n_calls)]:
        fn(c)

    per_row_repeat, call_times = [], []
    for _ in range(REPEATS):
        ts = np.empty(n_calls, dtype=np.float64)
        for i, c in enumerate(chunks):
            t0 = time.perf_counter_ns()
            fn(c)
            ts[i] = time.perf_counter_ns() - t0
        per_row_repeat.append(ts.sum() / (n_calls * batch) / 1000.0)
        call_times.append(ts)

    allc = np.concatenate(call_times) / 1000.0        # microseconds per call
    pr = np.array(per_row_repeat)
    return {
        "n_calls": n_calls,
        "median_us_per_row": float(np.median(pr)),
        "iqr_us_per_row": float(np.subtract(*np.percentile(pr, [75, 25]))),
        "min_us_per_row": float(pr.min()),
        "max_us_per_row": float(pr.max()),
        "call_p50_us": float(np.percentile(allc, 50)),
        "call_p95_us": float(np.percentile(allc, 95)),
        "call_p99_us": float(np.percentile(allc, 99)),
    }

## 8. Correctness Check

A cheaper path is only interesting if it returns the same answer. This runs once per
configuration and writes the result to its own file, so the paper can state that every
alternative path agreed with the library's own sklearn call to within a stated tolerance.

In [ ]:
def to_prob(out, n):
    if isinstance(out, list):
        out = out[0]
    if isinstance(out, list) and len(out) and isinstance(out[0], dict):
        return np.array([o[max(o)] for o in out], dtype=np.float64)
    arr = np.asarray(out)
    if arr.dtype == object:
        return np.array([o[max(o)] for o in arr], dtype=np.float64)
    return arr.reshape(n, -1)[:, -1].astype(np.float64)

GROUPS = {"lightgbm": ["lgb_sklearn", "lgb_booster", "lgb_onnx"],
          "xgboost":  ["xgb_sklearn", "xgb_dmatrix", "xgb_inplace", "xgb_onnx"],
          "catboost": ["cat_sklearn", "cat_onnx"]}

def check_correctness(runners, X_probe, tag):
    rows = []
    n = len(X_probe)
    for lib, paths in GROUPS.items():
        paths = [p for p in paths if p in runners]
        if not paths:
            continue
        ref = to_prob(runners[paths[0]](X_probe), n)
        for p in paths[1:]:
            d = float(np.max(np.abs(to_prob(runners[p](X_probe), n) - ref)))
            rows.append({"config": tag, "library": lib, "reference": paths[0],
                         "path": p, "max_abs_diff": d, "agrees": d < 1e-4})
    df = pd.DataFrame(rows)
    append_row(df, CORRECT_CSV)
    return df

## 9. Run the Experiment

Three sweeps share one loop. The main sweep varies tree count on the real dataset, the
feature sweep varies column count on synthetic data, and the thread sweep repeats the main
setting with both cores enabled.

Two details protect the measurements. Configurations run in a shuffled order, so if the
machine slows down halfway through the session the slowdown does not land on one serving
path in particular. And a fixed reference measurement is repeated every twenty
configurations, which gives a direct record of how much the machine drifted while the
experiment ran.

Progress prints as it goes. Safe to interrupt and re-run at any point.

In [ ]:
Xs_tr, ys_tr, Xs_te = split(Xs, ys)

jobs = []
for nt in TREE_COUNTS:
    jobs.append({"sweep": "trees", "dataset": "online_shoppers", "n_features": Xs.shape[1],
                 "n_trees": nt, "threads": 1})
for nf in FEATURE_SWEEP:
    jobs.append({"sweep": "features", "dataset": "synthetic", "n_features": nf,
                 "n_trees": TREE_COUNTS[0], "threads": 1})
for th in THREAD_COUNTS:
    if th != 1:
        jobs.append({"sweep": "threads", "dataset": "online_shoppers",
                     "n_features": Xs.shape[1], "n_trees": TREE_COUNTS[0],
                     "threads": th})

PATHS = ["lgb_sklearn", "lgb_booster", "lgb_onnx", "xgb_sklearn", "xgb_dmatrix",
         "xgb_inplace", "xgb_onnx", "cat_sklearn", "cat_onnx"]

done = set()
if os.path.exists(RESULTS_CSV):
    prev = pd.read_csv(RESULTS_CSV)
    done = set(zip(prev["config"], prev["path"], prev["batch"]))

planned = sum(1 for j in jobs for b in BATCHES for p in PATHS
              if (f"{j['dataset']}_f{j['n_features']}_t{j['n_trees']}_th{j['threads']}",
                  p, b) not in done)
print(f"run id {RUN_ID}: {len(done)} measurements already on disk, "
      f"{planned} still to take")
if planned == 0:
    print("\nNOTHING WILL BE MEASURED. This run id is already complete, so the "
          "\ndrift log will stay empty. Change RUN_ID in cell 6 and run again "
          "\nif you meant to re-measure.")
print()

# One reference model, trained once and held for the whole session. Host drift is
# tracked by re-measuring this same model at a fixed batch size throughout the run, so
# the drift log is comparable across every configuration rather than only within one.
print("training the persistent drift reference model")
REF_MODEL = CatBoostClassifier(iterations=100, depth=DEPTH, thread_count=1,
                               verbose=0, random_seed=SEED).fit(Xs_tr, ys_tr)
REF_BATCH = 64
def ref_runner(b):
    return REF_MODEL.predict_proba(b)

ref_log, since_ref, t_start = [], 0, time.time()

def log_drift(where):
    rm = measure(ref_runner, Xs_te, REF_BATCH)
    ref_log.append({"t_min": (time.time() - t_start) / 60, "during": where,
                    "ref_model": "catboost_100t_b64_persistent",
                    "ref_us": rm["median_us_per_row"]})
    try:
        pd.DataFrame(ref_log).to_csv(NOISE_CSV, index=False)
    except Exception as e:
        print("  noise log write failed:", type(e).__name__)

log_drift("session_start")

for job in jobs:
    tag = (f"{job['dataset']}_f{job['n_features']}_t{job['n_trees']}"
           f"_th{job['threads']}")
    todo = [(p, b) for b in BATCHES for p in PATHS if (tag, p, b) not in done]
    if not todo:
        print(f"[skip] {tag} already complete")
        continue

    print(f"\n=== {tag} ===")
    if job["dataset"] == "synthetic":
        Xj, yj = make_synth(job["n_features"])
        Xtr, ytr, Xte = split(Xj, yj)
    else:
        Xtr, ytr, Xte = Xs_tr, ys_tr, Xs_te

    t0 = time.time()
    runners = build_runners(Xtr, ytr, job["n_trees"], job["threads"], tag)
    print(f"  trained in {time.time()-t0:.0f}s, {len(runners)} paths")
    check_correctness(runners, Xte[:64], tag)

    todo = [(p, b) for p, b in todo if p in runners]
    random.Random(SEED).shuffle(todo)

    for p, b in todo:
        try:
            m = measure(runners[p], Xte, b)
        except Exception as e:
            print(f"  {p:12s} b={b:6d}  FAILED: {type(e).__name__} {str(e)[:70]}")
            continue
        rec = {"config": tag, **job, "path": p, "batch": b, **m}
        append_row(rec, RESULTS_CSV)
        since_ref += 1
        if since_ref >= 20:
            since_ref = 0
            log_drift(tag)
            sync_to_drive()
        print(f"  {p:12s} b={b:6d}  {m['median_us_per_row']:9.2f} us/row  "
              f"(p95 call {m['call_p95_us']:.1f} us, {m['n_calls']} calls)")

    sync_to_drive(quiet=False)

log_drift("session_end")
print(f"\nfinished in {(time.time()-t_start)/60:.1f} minutes")
sync_to_drive(quiet=False)
res = pd.read_csv(RESULTS_CSV).drop_duplicates(
    subset=["config", "path", "batch"], keep="last")
print(f"{len(res)} measurements saved to {RESULTS_CSV}")

## 10. Result Tables

Table 1 is the main latency matrix. Table 2 is the penalty ratio, which is the headline
number. Table 3 splits the batch-size-one cost into the part that is the Python interface
and the part that is real inference work.

In [ ]:
res = pd.read_csv(RESULTS_CSV).drop_duplicates(
    subset=["config", "path", "batch"], keep="last")
MAIN = f"online_shoppers_f{Xs.shape[1]}_t{TREE_COUNTS[-1] if len(TREE_COUNTS)>1 else TREE_COUNTS[0]}_th1"
main = res[res["config"] == MAIN]
if main.empty:
    MAIN = res["config"].iloc[0]
    main = res[res["config"] == MAIN]

print("Table 1. Per-row latency in microseconds, configuration:", MAIN)
t1 = main.pivot(index="path", columns="batch", values="median_us_per_row")
display(t1.round(2))

print("\nTable 2. Batch-of-one penalty, smallest batch over largest batch")
lo, hi = t1.columns.min(), t1.columns.max()
t2 = pd.DataFrame({"batch1_us": t1[lo], "batchmax_us": t1[hi],
                   "penalty": t1[lo] / t1[hi],
                   "p95_call_us_at_b1": main[main["batch"] == lo]
                       .set_index("path")["call_p95_us"]})
display(t2.sort_values("penalty", ascending=False).round(2))

print("\nTable 3. Share of the batch-one cost attributable to the Python interface")
pairs = [("lightgbm", "lgb_sklearn", "lgb_booster"),
         ("xgboost", "xgb_sklearn", "xgb_inplace"),
         ("catboost", "cat_sklearn", "cat_onnx")]
rows = []
for lib, w, n in pairs:
    if w in t1.index and n in t1.index:
        wv, nv = t1.loc[w, lo], t1.loc[n, lo]
        rows.append({"library": lib, "wrapper_path": w, "wrapper_us": wv,
                     "leaner_path": n, "leaner_us": nv,
                     "overhead_us": wv - nv,
                     "overhead_share_pct": 100 * (wv - nv) / wv})
display(pd.DataFrame(rows).round(2))

## 11. Figures

Four figures, saved as PNG to the `figures` folder in Drive at 300 dpi. These are drafts
for placement and sizing. Redraw them yourself before submission.

In [ ]:
def save(fig, name):
    p = os.path.join(FIG_DIR, name)
    fig.savefig(p, dpi=300, bbox_inches="tight")
    print("saved", p)

# Figure 1. latency against batch size
fig, ax = plt.subplots(figsize=(7, 5))
for p in sorted(t1.index):
    ax.plot(t1.columns, t1.loc[p], marker="o", ms=4, label=p)
ax.set_xscale("log", base=2); ax.set_yscale("log")
ax.set_xlabel("Batch size (rows per call)")
ax.set_ylabel("Latency per row (microseconds)")
ax.grid(True, which="both", alpha=0.3); ax.legend(fontsize=8, ncol=2)
save(fig, "fig1_latency_vs_batch.png"); plt.show()

# Figure 2. penalty ratio
fig, ax = plt.subplots(figsize=(7, 4))
s = (t1[lo] / t1[hi]).sort_values()
ax.barh(s.index, s.values)
ax.set_xscale("log")
ax.set_xlabel(f"Penalty ratio, batch {lo} over batch {hi}")
ax.grid(True, axis="x", alpha=0.3)
save(fig, "fig2_penalty_ratio.png"); plt.show()

# Figure 3. penalty against tree count
tw = res[(res["sweep"] == "trees") & (res["batch"].isin([lo, hi]))]
if tw["n_trees"].nunique() > 1:
    fig, ax = plt.subplots(figsize=(7, 4))
    for p, g in tw.groupby("path"):
        pv = g.pivot(index="n_trees", columns="batch", values="median_us_per_row")
        if lo in pv.columns and hi in pv.columns:
            ax.plot(pv.index, pv[lo] / pv[hi], marker="o", label=p)
    ax.set_yscale("log")
    ax.set_xlabel("Number of trees"); ax.set_ylabel("Penalty ratio")
    ax.grid(True, alpha=0.3); ax.legend(fontsize=8, ncol=2)
    save(fig, "fig3_penalty_vs_trees.png"); plt.show()
else:
    print("tree sweep needs more than one tree count, skipping figure 3")

# Figure 4. machine drift during the session
nref = NOISE_CSV
if os.path.exists(nref):
    nd = pd.read_csv(nref)
    fig, ax = plt.subplots(figsize=(7, 3.5))
    ax.plot(nd["t_min"], nd["ref_us"], marker="o", ms=4)
    ax.set_xlabel("Minutes into the session")
    ax.set_ylabel("Reference latency (us/row)")
    ax.grid(True, alpha=0.3)
    save(fig, "fig4_machine_drift.png"); plt.show()
    d = nd["ref_us"]
    print(f"drift across session: {100*(d.max()-d.min())/d.median():.1f}% "
          f"of the median")
sync_to_drive(quiet=False)

## 11b. Crossover Robustness

The exact batch size at which one serving path overtakes another cannot be pinned down
precisely, because at a crossover the two paths cost roughly the same and the difference
falls inside the measurement noise. What can be established is the regime on either side.

This cell compares the winning margin at each batch size against the noise on the winning
measurement, and marks a winner as robust only when it leads by more than three times the
interquartile range. The paper should claim only the robust rows.

In [ ]:
LIBS = {"lightgbm": ["lgb_sklearn", "lgb_booster", "lgb_onnx"],
        "xgboost":  ["xgb_sklearn", "xgb_dmatrix", "xgb_inplace", "xgb_onnx"],
        "catboost": ["cat_sklearn", "cat_onnx"]}

rob = []
tw = res[res["sweep"] == "trees"]
for nt in sorted(tw["n_trees"].unique()):
    for lib, paths in LIBS.items():
        for b in sorted(tw["batch"].unique()):
            g = tw[(tw["n_trees"] == nt) & (tw["batch"] == b) &
                   (tw["path"].isin(paths))].set_index("path")
            if len(g) < 2:
                continue
            srt = g["median_us_per_row"].sort_values()
            best = srt.index[0]
            margin = 100 * (srt.iloc[1] - srt.iloc[0]) / srt.iloc[0]
            noise = 100 * g.loc[best, "iqr_us_per_row"] / g.loc[best, "median_us_per_row"]
            rob.append({"n_trees": nt, "library": lib, "batch": b, "winner": best,
                        "margin_pct": margin, "iqr_pct": noise,
                        "robust": margin > 3 * noise})
rob = pd.DataFrame(rob)
rob.to_csv(os.path.join(LOCAL_DIR, "crossover_robustness.csv"), index=False)

print("Robust winners only (margin greater than three times the noise):")
display(rob[rob["robust"]].pivot_table(index=["library", "n_trees"], columns="batch",
                                       values="winner", aggfunc="first"))
print(f"\n{rob['robust'].mean()*100:.0f}% of comparisons have a robust winner")

## 11c. CatBoost ONNX Ablation

CatBoost's ONNX export ends in a ZipMap node, which builds a Python dictionary for every
row of output. That raises an obvious question: is the ONNX path's bulk cost measuring
inference, or is it measuring output construction?

This cell removes the node by editing the graph directly and re-measures. Running it inside
the experiment rather than separately means the answer is reproducible from the repository,
which is where a reviewer will look for it.

In [ ]:
import onnx
from onnx import helper

try:
    ref_cb = CatBoostClassifier(iterations=TREE_COUNTS[-1], depth=DEPTH, thread_count=1,
                                verbose=0, random_seed=SEED).fit(Xs_tr, ys_tr)
    p_zip = os.path.join(LOCAL_DIR, "ablation_with_zipmap.onnx")
    ref_cb.save_model(p_zip, format="onnx")

    g = onnx.load(p_zip)
    zm = [n for n in g.graph.node if n.op_type == "ZipMap"][0]
    raw = zm.input[0]
    g.graph.node.remove(zm)
    while len(g.graph.output):
        g.graph.output.pop()
    g.graph.output.extend([helper.make_tensor_value_info(
        raw, onnx.TensorProto.FLOAT, [None, 2])])
    p_nozip = os.path.join(LOCAL_DIR, "ablation_without_zipmap.onnx")
    onnx.save(g, p_nozip)

    so_a = ort.SessionOptions()
    so_a.intra_op_num_threads = 1
    so_a.inter_op_num_threads = 1

    def onnx_runner(path):
        sess = ort.InferenceSession(path, so_a, providers=["CPUExecutionProvider"])
        i, o = sess.get_inputs()[0].name, sess.get_outputs()[-1].name
        return lambda b, s=sess, i=i, o=o: s.run([o], {i: b})

    abl = []
    for b in [1, 64, 1024, 16384]:
        w = measure(onnx_runner(p_zip), Xs_te, b)["median_us_per_row"]
        n = measure(onnx_runner(p_nozip), Xs_te, b)["median_us_per_row"]
        abl.append({"batch": b, "with_zipmap_us": w, "without_zipmap_us": n,
                    "ratio": w / n})
        print(f"batch {b:6d}  with ZipMap {w:8.2f}  without {n:8.2f}  ratio {w/n:.2f}x")
    abl = pd.DataFrame(abl)
    abl.to_csv(os.path.join(LOCAL_DIR, "zipmap_ablation.csv"), index=False)
    print("\nA ratio near 1.0 means ZipMap is not responsible and the ONNX cost "
          "is real inference cost.")
except Exception as e:
    print("ablation failed:", type(e).__name__, str(e)[:150])

## 12. Export Everything for the Paper

Bundles the tables into one Excel file next to the figures, so the whole result set is one
download rather than a hunt through Drive.

In [ ]:
out = os.path.join(SAVE_DIR, "paper_tables.xlsx")
with pd.ExcelWriter(out) as w:
    t1.round(3).to_excel(w, sheet_name="T1_latency")
    t2.round(3).to_excel(w, sheet_name="T2_penalty")
    pd.DataFrame(rows).round(3).to_excel(w, sheet_name="T3_overhead", index=False)
    res.to_excel(w, sheet_name="all_measurements", index=False)
    for _n, _sheet in [("crossover_robustness.csv", "crossover_robustness"),
                       ("zipmap_ablation.csv", "zipmap_ablation"),
                       ("noise_reference.csv", "drift_log")]:
        _p = os.path.join(LOCAL_DIR, _n)
        if os.path.exists(_p):
            pd.read_csv(_p).to_excel(w, sheet_name=_sheet, index=False)
    if os.path.exists(CORRECT_CSV):
        pd.read_csv(CORRECT_CSV).drop_duplicates().to_excel(
            w, sheet_name="correctness", index=False)
print("written:", out)
sync_to_drive(quiet=False)

cc = pd.read_csv(CORRECT_CSV).drop_duplicates() if os.path.exists(CORRECT_CSV) else None
if cc is not None:
    bad = cc[~cc["agrees"]]
    print(f"\ncorrectness: {len(cc)} comparisons, {len(bad)} disagreements")
    if len(bad):
        display(bad)
    else:
        print("every alternative path matched its library's sklearn call "
              "within 1e-4")

## If the Run Stops

Storage problems can no longer stop it, but a Colab session can still time out. Recovery is
the same every time: **Runtime → Run all**. The notebook reads what is already saved,
prints how many measurements it found, and carries on from there.

If Drive was unmounted when the session died, the recovery step in cell 8 pulls the results
back off Drive before the loop starts, so nothing measured is ever measured twice.

## What to Send Me

The Excel file and the four PNGs. With those I can draft the paper sections.

Before you close the notebook, sanity check three things.

Figure 4 should be roughly flat. A visible upward drift means the machine got slower during
the run, and while the shuffled ordering stops that from biasing any one path, it does need
mentioning in the threats to validity.

The correctness count at the bottom should report zero disagreements. If a path disagrees,
its latency numbers cannot be used and the paper has to say so.

In Table 2 the interquartile range should be small next to the gaps between paths. If they
are the same size, raise `REPEATS` to 21 and run the batch-size-one rows again.